# ECG Analysis — DACIL-WESENSE

Loads WESENSE BDF files with **MNE-Python**, reads each sensor recording,
extracts time-resolved cardiopulmonary features per 30-second window, and
produces per-patient and cohort-level visualisations.

**Features extracted per window:**
| Feature | Method |
|---|---|
| Heart rate (bpm) | Median RR interval from R-peak detection |
| RMSSD (ms) | Square root of mean squared successive RR differences |
| SDNN (ms) | Standard deviation of RR intervals |
| LF power (ms²) | Spectral power 0.04–0.15 Hz (exploratory at 30 s) |
| HF power (ms²) | Spectral power 0.15–0.40 Hz |
| LF/HF ratio | Sympatho-vagal balance proxy |
| Breathing rate (bpm) | ECG-derived respiration (EDR) via R-peak amplitude modulation |


In [ ]:
# TODO: request the data in a more sensible format than BDF files because they are a fucking pain to work with, they're so god damn outdated I cannot take this anymore.

In [ ]:
# ── Standard library ───────────────────────────────────────────────────────
from pathlib import Path

# ── Third-party ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mne  # BDF file reader via MNE-Python

# ── Project helpers ────────────────────────────────────────────────────────
import functions as fn

# ── Configuration ──────────────────────────────────────────────────────────
DATA_ROOT   = "data"
OUTPUT_ROOT = "output"

# Folder name of the patient to use for the single-patient deep-dive.
# Set to None to auto-select the first available patient.
PATIENT_ID: str | None = None

# Window length for feature extraction (seconds).
WINDOW_DURATION: float = 30.0

In [ ]:
fn.setup_logging(log_file="pipeline.log")

---
## Helper: load BDF with MNE

`fn.load_ecg()` uses MNE's `read_raw_bdf()` to open a BDF file and returns
an `mne.io.BaseRaw` object (lazy-loaded, no full array allocation needed).


---
## Part A — Single-patient deep-dive

In [ ]:
# ── Discover patient folders ───────────────────────────────────────────────
all_folders = fn.discover_patient_folders(DATA_ROOT)
if not all_folders:
    raise RuntimeError(f"No patient folders found under '{DATA_ROOT}'")

if PATIENT_ID is not None:
    folder = next((f for f in all_folders if f.name == PATIENT_ID), None)
    if folder is None:
        raise ValueError(f"Patient '{PATIENT_ID}' not found in '{DATA_ROOT}'")
else:
    folder = all_folders[0]
    PATIENT_ID = folder.name

print(f"Patient: {PATIENT_ID}")
out_dir = Path(OUTPUT_ROOT) / PATIENT_ID
out_dir.mkdir(parents=True, exist_ok=True)

# ── Find BDF files ─────────────────────────────────────────────────────────
l1_path, l2_path = fn.find_bdf_files(folder)
print(f"L1: {l1_path.name if l1_path else 'not found'}")
print(f"L2: {l2_path.name if l2_path else 'not found'}")

if l1_path is None and l2_path is None:
    raise RuntimeError(f"No BDF files found for patient '{PATIENT_ID}'")

In [ ]:
# ── Load BDF files via MNE ────────────────────────────────────────────────
sensors = {}
for label, path in [("L1", l1_path), ("L2", l2_path)]:
    if path is None:
        continue
    raw = fn.load_ecg(path)
    if raw is None:
        print(f"  {label}: failed to load — skipping")
        continue
    sensors[label] = {"raw": raw}

# Show duration and channel names for each sensor
for label, s in sensors.items():
    r = s["raw"]
    print(f"  {label}: {r.n_times} samples @ {r.info['sfreq']:.0f} Hz, "
          f"duration {r.times[-1]:.1f} s, channels: {r.ch_names}")


In [ ]:
# ── Extract time-resolved ECG features ────────────────────────────────────
#
# fn.extract_ecg_timeseries() accepts an MNE Raw object and applies
# sliding-window R-peak detection, HRV and EDR analysis.

ts_frames = []
for label, s in sensors.items():
    print(f"Extracting features for sensor {label} …")
    ts = fn.extract_ecg_timeseries(
        s["raw"],
        window_duration=WINDOW_DURATION,
    )
    ts.insert(0, "sensor", label)
    ts_frames.append(ts)

ts_all = pd.concat(ts_frames, ignore_index=True)
print(f"\nTimeseries shape: {ts_all.shape}")
ts_all.head()


### Heart rate over time

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"], sort=False):
    t_min = grp["time_s"] / 60
    ax.plot(t_min, grp["hr_bpm"], label=f"{sensor} – {channel}", linewidth=1.2)
    mean_hr = grp["hr_bpm"].mean()
    if not np.isnan(mean_hr):
        ax.axhline(mean_hr, linestyle="--", linewidth=0.8, alpha=0.55,
                   color=ax.lines[-1].get_color())

ax.set_xlabel("Time (min)")
ax.set_ylabel("Heart rate (bpm)")
ax.set_title(f"{PATIENT_ID} — Heart rate")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_hr.png")
plt.show()

### HRV — time-domain (RMSSD & SDNN)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"], sort=False):
    label = f"{sensor} – {channel}"
    t_min = grp["time_s"] / 60
    axes[0].plot(t_min, grp["rmssd_ms"], label=label, linewidth=1.2)
    axes[1].plot(t_min, grp["sdnn_ms"],  label=label, linewidth=1.2)

for ax, metric in zip(axes, ["RMSSD (ms)", "SDNN (ms)"]):
    ax.set_ylabel(metric)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_title(f"{PATIENT_ID} — HRV time-domain")
axes[1].set_xlabel("Time (min)")
fig.tight_layout()
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_hrv_timedomain.png")
plt.show()

### HRV — frequency-domain (LF, HF, LF/HF)

> Values are exploratory at the default 30-second window length. The clinical standard requires ≥ 5-minute windows.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
metrics_freq = [
    ("lf_ms2",      "LF power (ms²)"),
    ("hf_ms2",      "HF power (ms²)"),
    ("lf_hf_ratio", "LF/HF ratio"),
]

for (sensor, channel), grp in ts_all.groupby(["sensor", "channel"], sort=False):
    label = f"{sensor} – {channel}"
    t_min = grp["time_s"] / 60
    for ax, (col, _) in zip(axes, metrics_freq):
        ax.plot(t_min, grp[col], label=label, linewidth=1.2)

axes[0].set_title(f"{PATIENT_ID} — HRV frequency-domain (exploratory at {WINDOW_DURATION:.0f} s windows)")
for ax, (_, ylabel) in zip(axes, metrics_freq):
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (min)")
fig.tight_layout()
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_hrv_freqdomain.png")
plt.show()

### Breathing rate over time (ECG-derived respiration)

Estimated from the amplitude modulation of R-peaks — a well-established ECG-derived respiration (EDR) proxy. Cross-validate against the VE column from telemetry data where available.

### ECG raw signal preview (first 10 seconds)

Visual quality check of the raw signal and detected R-peaks.

In [ ]:
PREVIEW_SECONDS = 10

n_panels = sum(len(s["raw"].ch_names) for s in sensors.values())
fig, axes = plt.subplots(n_panels, 1, figsize=(13, 3 * n_panels), sharex=True)
if n_panels == 1:
    axes = [axes]

panel = 0
for sensor_label, s in sensors.items():
    raw = s["raw"]
    sfreq = raw.info["sfreq"]
    n_show = int(PREVIEW_SECONDS * sfreq)
    t_show = np.arange(n_show) / sfreq

    for ch_idx, ch_name in enumerate(raw.ch_names):
        sig = raw[ch_idx, :n_show][0].squeeze().astype(float)

        # Detect R-peaks in the preview segment only.
        r_peaks = fn._detect_r_peaks(sig, sfreq)

        ax = axes[panel]
        ax.plot(t_show, sig, linewidth=0.8, color="steelblue", label="ECG")
        if len(r_peaks):
            ax.scatter(r_peaks / sfreq, sig[r_peaks], color="crimson", s=25,
                       zorder=3, label="R-peaks")
        ax.set_ylabel("Amplitude")
        ax.set_title(f"{sensor_label} – {ch_name}")
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True, alpha=0.3)
        panel += 1

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"{PATIENT_ID} — Raw ECG preview (first {PREVIEW_SECONDS} s)", fontsize=12)
fig.tight_layout()
fn._save_figure(fig, out_dir, f"{PATIENT_ID}_ecg_raw_preview.png")
plt.show()


### Summary statistics for this patient

In [ ]:
summary_cols = ["hr_bpm", "rmssd_ms", "sdnn_ms", "lf_ms2", "hf_ms2",
                "lf_hf_ratio", "breathing_rate_bpm"]
ts_all[summary_cols].describe().round(2)

---
## Part B — Batch summary across all patients

In [ ]:
summary_rows = []

for fold in all_folders:
    pid = fold.name
    l1, l2 = fn.find_bdf_files(fold)

    frames = []
    for sensor_label, bdf_path in [("L1", l1), ("L2", l2)]:
        if bdf_path is None:
            continue
        raw = fn.load_ecg(bdf_path)
        if raw is None:
            print(f"  {pid} {sensor_label}: load failed — skipping")
            continue
        ts = fn.extract_ecg_timeseries(raw, window_duration=WINDOW_DURATION)
        ts.insert(0, "sensor", sensor_label)
        frames.append(ts)

    if not frames:
        print(f"  {pid}: no usable BDF data — skipping")
        continue

    ts_pat = pd.concat(frames, ignore_index=True)
    summary_rows.append(
        dict(
            patient_id=pid,
            mean_hr_bpm        = ts_pat["hr_bpm"].mean(),
            mean_rmssd_ms      = ts_pat["rmssd_ms"].mean(),
            mean_sdnn_ms       = ts_pat["sdnn_ms"].mean(),
            mean_lf_ms2        = ts_pat["lf_ms2"].mean(),
            mean_hf_ms2        = ts_pat["hf_ms2"].mean(),
            mean_lf_hf         = ts_pat["lf_hf_ratio"].mean(),
            mean_breathing_bpm = ts_pat["breathing_rate_bpm"].mean(),
        )
    )
    print(f"  {pid}: done")

batch_df = pd.DataFrame(summary_rows)
print(f"\nBatch complete: {len(batch_df)} patients")
batch_df


In [ ]:
metrics_batch = [
    ("mean_hr_bpm",         "Mean HR (bpm)"),
    ("mean_rmssd_ms",       "Mean RMSSD (ms)"),
    ("mean_sdnn_ms",        "Mean SDNN (ms)"),
    ("mean_lf_ms2",         "Mean LF power (ms²)"),
    ("mean_hf_ms2",         "Mean HF power (ms²)"),
    ("mean_lf_hf",          "Mean LF/HF ratio"),
    ("mean_breathing_bpm",  "Mean breathing rate\n(bpm, EDR)"),
]

n = len(metrics_batch)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))

for ax, (col, ylabel) in zip(axes, metrics_batch):
    vals = batch_df[col].dropna().values
    if len(vals):
        ax.boxplot(vals, widths=0.5)
        ax.scatter(np.ones(len(vals)), vals, color="steelblue", s=30,
                   zorder=3, alpha=0.75)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_xticks([])
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("ECG feature summary — all patients", fontsize=12)
fig.tight_layout()

batch_out = Path(OUTPUT_ROOT)
batch_out.mkdir(parents=True, exist_ok=True)
fn._save_figure(fig, batch_out, "ecg_batch_summary.png")
plt.show()